In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
import os
import sys

In [0]:
current_dir = os.getcwd()
display(current_dir)
sys.path.append(current_dir)

### **CUSTOMERS**

In [0]:
df_cust = spark.read.table("pysparkdbt.bronze.customers")


In [0]:
df_cust.count()

#### **Basic Data Transformation Operations**

In [0]:
df_cust =  df_cust.withColumn("domain",split(col('email'),'@')[1])


In [0]:
df_cust = df_cust.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))


In [0]:
df_cust = df_cust.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))
df_cust = df_cust.drop('first_name','last_name')


In [0]:
from utils.custom_utils import var_1


In [0]:
var_1

#### **De-Duplication**

In [0]:
from utils.custom_utils import transformations

In [0]:
cust_obj = transformations()
cust_df_trns = cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')


#### **Upserts**

In [0]:
df_cust = cust_obj.process_timestamp(cust_df_trns)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):

    df_cust.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.customers")

else:
    cust_obj.upsert(df_cust,['customer_id'],'customers','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.customers

### **DRIVERS**

In [0]:
df_driver = spark.read.table("pysparkdbt.bronze.drivers")
display(df_driver)


### Basic Data Transformation Operations

In [0]:
df_driver = df_driver.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))


In [0]:
df_driver = df_driver.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))
df_driver = df_driver.drop('first_name','last_name')


### De-Duplication

In [0]:
driver_obj = transformations()
driver_df_trns = driver_obj.dedup(df_driver,['driver_id'],'last_updated_timestamp')



### Upserts

In [0]:
df_driver = driver_obj.process_timestamp(driver_df_trns)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):

    df_driver.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.drivers")

else:
    driver_obj.upsert(df_driver,['driver_id'],'drivers','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.drivers

### **LOCATIONS**

In [0]:
df_loc = spark.read.table("pysparkdbt.bronze.locations")
display(df_loc)

### De-Duplication

In [0]:
loc_obj = transformations()
loc_df_trns = loc_obj.dedup(df_loc,['location_id'],'last_updated_timestamp')


### Upserts

In [0]:
df_loc = loc_obj.process_timestamp(loc_df_trns)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):
    df_loc.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.locations")
else:
    loc_obj.upsert(df_loc,['location_id'],'locations','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.locations

### **PAYMENTS**

In [0]:
df_pay = spark.read.table("pysparkdbt.bronze.payments")
display(df_pay)

### Basic Data Transformation Operations

In [0]:
df_pay = df_pay.withColumn("online_payment_status",
               when( ((col('payment_method')=='Card') & (col('payment_status')=='Success')),"online-success")
               .when( ((col('payment_method')=='Card') & (col('payment_status')=='Failed')),"online-failed")
               .when( ((col('payment_method')=='Card') & (col('payment_status')=='Pending')),"online-pending")
               .otherwise("offline")
               )

display(df_pay)

### De-Duplication

In [0]:
payment_obj = transformations()
payment_df_trns = payment_obj.dedup(df_pay,['payment_id'],'last_updated_timestamp')

### Upserts

In [0]:
df_pay = payment_obj.process_timestamp(payment_df_trns)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):
    df_pay.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.payments")
else:
    payment_obj.upsert(df_pay,['payment_id'],'payments','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.payments

### **VEHICLES**

In [0]:
df_veh = spark.read.table("pysparkdbt.bronze.vehicles")
display(df_veh)

### Basic Data Transformation Operations

In [0]:
df_veh = df_veh.withColumn("make",upper(col("make")))
display(df_veh)

### De-Duplication

In [0]:
veh_obj = transformations()
veh_df_trns = veh_obj.dedup(df_veh,['vehicle_id'],'last_updated_timestamp')

### Upserts

In [0]:
df_veh = veh_obj.process_timestamp(veh_df_trns)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):
    df_veh.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.vehicles")
else:
    veh_obj.upsert(df_veh,['vehicle_id'],'vehicles','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.vehicles